In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (roc_curve, auc, precision_recall_curve, 
                             accuracy_score, precision_score, recall_score, f1_score)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# ==========================================
# 0. 环境设置
# ==========================================
sns.set_style("whitegrid")
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings("ignore")

SEED = 0
np.random.seed(SEED)

# ==========================================
# 1. 数据准备
# ==========================================
print("正在读取第一阶段数据...")
# 读取入科前/入科即刻数据
df_before = pd.read_csv('全部病房患者/内部训练集/preicu.csv')

# --- 特征定义 (仅使用 Stage 1 特征) ---
cols_stage1 = [
    'gender', 'age', 'admissionroute', 
    'temp_max', 'temp_min', 'resp_max', 'resp_min',
    'hr_max', 'hr_min', 'map_max', 'map_min',
    'tbil', 'glu', 'cre', 'plt', 'wbc', 'lym', 
    'vasopressor', 'MV'
]

# 检查特征是否存在
missing_cols = [c for c in cols_stage1 if c not in df_before.columns]
if missing_cols:
    raise ValueError(f"以下特征在数据中未找到: {missing_cols}")

X = df_before[cols_stage1]
y = df_before['death_90d'].values

print(f"样本数: {len(X)}, 特征数: {len(cols_stage1)}")
print(f"正样本比例: {np.mean(y):.2%}")

# ==========================================
# 2. 五折交叉验证 (Logistic Regression)
# ==========================================
print("\n--- Starting 5-Fold CV (Stage 1 Logistic Regression) ---")

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

cv_metrics = {'auc': [], 'acc': [], 'prec': [], 'recall': [], 'f1': []}
tprs = []
mean_fpr = np.linspace(0, 1, 100)
fold_no = 1

for train_idx, val_idx in kfold.split(X, y):
    
    # 切分数据
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # 预处理：逻辑回归对数据尺度敏感，必须进行标准化
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    # 构建模型
    model = LogisticRegression(
        class_weight='balanced', 
        solver='liblinear', 
        random_state=SEED,
        C=1.0 # 正则化强度，默认1.0
    )
    
    # 训练
    model.fit(X_train_scaled, y_train)
    
    # 预测概率
    y_pred_prob = model.predict_proba(X_val_scaled)[:, 1]
    
    # --- 计算指标 ---
    
    # 1. AUC
    fpr, tpr, _ = roc_curve(y_val, y_pred_prob)
    roc_auc = auc(fpr, tpr)
    cv_metrics['auc'].append(roc_auc)
    
    # 用于绘制平均 ROC 曲线
    tprs.append(np.interp(mean_fpr, fpr, tpr))
    tprs[-1][0] = 0.0
    
    # 2. 寻找最佳 F1 阈值 (Best Threshold)
    # 因为是不平衡数据，默认 0.5 阈值通常不是最优的
    precisions, recalls, thresholds = precision_recall_curve(y_val, y_pred_prob)
    with np.errstate(divide='ignore', invalid='ignore'):
        f1_scores_curve = 2 * (precisions * recalls) / (precisions + recalls)
    f1_scores_curve = np.nan_to_num(f1_scores_curve)
    
    best_idx = np.argmax(f1_scores_curve)
    best_f1 = f1_scores_curve[best_idx]
    best_thresh = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    
    # 基于最佳阈值生成预测类别
    y_pred_class = (y_pred_prob >= best_thresh).astype(int)
    
    # 计算其他指标
    acc = accuracy_score(y_val, y_pred_class)
    prec = precision_score(y_val, y_pred_class, zero_division=0)
    rec = recall_score(y_val, y_pred_class, zero_division=0)
    
    cv_metrics['acc'].append(acc)
    cv_metrics['prec'].append(prec)
    cv_metrics['recall'].append(rec)
    cv_metrics['f1'].append(best_f1)
    
    print(f"Fold {fold_no} -> AUC: {roc_auc:.4f} | F1: {best_f1:.4f} | Acc: {acc:.4f} (Thresh: {best_thresh:.3f})")
    fold_no += 1

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (roc_curve, auc, precision_recall_curve, 
                             accuracy_score, precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# ==========================================
# 0. 环境设置
# ==========================================
sns.set_style("whitegrid")
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings("ignore")

SEED = 0
np.random.seed(SEED)

# ==========================================
# 1. 数据准备
# ==========================================
print("正在读取数据...")
df_before = pd.read_csv('全部病房患者/内部训练集/preicu.csv')
df_after = pd.read_csv('全部病房患者/内部训练集/icu.csv') 

cols_stage1 = [
    'gender', 'age', 'admissionroute', 
    'temp_max', 'temp_min', 'resp_max', 'resp_min',
    'hr_max', 'hr_min', 'map_max', 'map_min',
    'tbil', 'glu', 'cre', 'plt', 'wbc', 'lym', 
    'vasopressor', 'MV'
]

cols_stage2_raw = [
    'gender', 'age', 'admissionroute',
    'temp_max', 'temp_min', 'resp_max', 'resp_min',
    'hr_max', 'hr_min', 'map_max', 'map_min',
    'glu', 'tbil', 'cre', 'spo2', 'plt',
    'wbc', 'lym', 'ph', 'lac',
    'vasopressor', 'MV'
]

binary_cols = ['gender', 'vasopressor', 'MV', 'admissionroute']
continuous_cols = [c for c in cols_stage2_raw if c not in binary_cols]

y_all = df_before['death_90d'].values
indices = np.arange(len(df_before))

assert len(df_before) == len(df_after), "两份数据文件行数不一致！"

# ==========================================
# 定义超参数搜索包装函数
# ==========================================
def optimize_model(estimator, param_dist, name, X, y):
    print(f"    正在优化 {name} ...")
    search = RandomizedSearchCV(
        estimator, param_distributions=param_dist, 
        n_iter=100, # 建议测试时改小该值，如 n_iter=10
        scoring='roc_auc', cv=5, n_jobs=-1, random_state=SEED, verbose=0
    )
    search.fit(X, y)
    return search.best_estimator_

# ==========================================
# 2. 五折交叉验证 (Stacking Ensemble)
# ==========================================
print("\n--- Starting 5-Fold CV (Stacking Ensemble with Hyperparameter Tuning) ---")
print("基模型列表: XGBoost, LightGBM, Random Forest, GaussianNB, SVM") 
print("元模型: Logistic Regression")

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

cv_metrics = {'auc': [], 'acc': [], 'prec': [], 'recall': [], 'f1': []}
tprs = []
mean_fpr = np.linspace(0, 1, 100)
fold_no = 1

for train_idx, val_idx in kfold.split(indices, y_all):
    print(f"\n========== Fold {fold_no} ==========")
    
    # -----------------------------------
    # A. 第一阶段: LR 生成风险分
    # -----------------------------------
    X_s1_train = df_before.iloc[train_idx][cols_stage1]
    y_train = y_all[train_idx]
    X_s1_val = df_before.iloc[val_idx][cols_stage1]
    y_val = y_all[val_idx]
    
    scaler_s1 = RobustScaler()
    X_s1_train_scaled = scaler_s1.fit_transform(X_s1_train)
    X_s1_val_scaled = scaler_s1.transform(X_s1_val)
    
    lr_s1 = LogisticRegression(class_weight='balanced', solver='liblinear', random_state=SEED)
    lr_s1.fit(X_s1_train_scaled, y_train)
    
    risk_train = lr_s1.predict_proba(X_s1_train_scaled)[:, 1]
    risk_val = lr_s1.predict_proba(X_s1_val_scaled)[:, 1]
    
    # -----------------------------------
    # B. 第二阶段: 数据预处理
    # -----------------------------------
    X_s2_train_raw = df_after.iloc[train_idx][cols_stage2_raw].copy()
    X_s2_val_raw = df_after.iloc[val_idx][cols_stage2_raw].copy()
    
    X_s2_train_raw['pre_icu_risk'] = risk_train
    X_s2_val_raw['pre_icu_risk'] = risk_val
    
    current_cont_cols = continuous_cols + ['pre_icu_risk']
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('cont', RobustScaler(), current_cont_cols),
            ('cat', 'passthrough', binary_cols)
        ],
        verbose_feature_names_out=False
    )
    
    preprocessor.set_output(transform='pandas')
    
    X_train_final = preprocessor.fit_transform(X_s2_train_raw)
    X_val_final = preprocessor.transform(X_s2_val_raw)
    
    # -----------------------------------
    # C. 超参数调优 (RandomizedSearchCV)
    # -----------------------------------
    print("  --- Hyperparameter Optimization ---")
    pos_count = np.sum(y_train == 1)
    neg_count = np.sum(y_train == 0)
    scale_pos_weight = neg_count / pos_count if pos_count > 0 else 1.0

    # 参数空间定义 (根据当前折的数据比例更新)
    xgb_params = {
        'n_estimators': [300, 500, 700],
        'learning_rate': [0.01],
        'max_depth': [3, 4, 5],
        'subsample': [0.7, 0.8, 0.9],
        'colsample_bytree': [0.7, 0.8, 0.9],
        'min_child_weight': [2, 3, 4],
        'scale_pos_weight': [scale_pos_weight]
    }

    lgb_params = {
        'n_estimators': [300, 500, 700],
        'num_leaves': [15, 31],
        'learning_rate': [0.01],
        'max_depth': [4, 5, 6],
        'subsample': [0.7, 0.8, 0.9],
        'min_child_samples': [10, 20],
        'scale_pos_weight': [scale_pos_weight]
    }

    rf_params = {
        'n_estimators': [300, 500, 700],
        'max_depth': [5, 10],
        'min_samples_split': [3, 4, 5],
        'min_samples_leaf': [2, 3, 4],
        'max_features': ['sqrt'],
        'class_weight': ['balanced']
    }

    svm_params = {
        'C': [0.5, 1.0, 2.0],
        'kernel': ['rbf', 'linear'],
        'gamma': ['scale', 'auto']
    }
    
    # 依次优化寻找本折的最优模型
    best_xgb = optimize_model(xgb.XGBClassifier(eval_metric='auc', random_state=SEED, n_jobs=1), 
                              xgb_params, "XGBoost", X_train_final, y_train)
    best_lgbm = optimize_model(lgb.LGBMClassifier(objective='binary', metric='auc', verbosity=-1, random_state=SEED, n_jobs=1), 
                               lgb_params, "LightGBM", X_train_final, y_train)
    best_rf = optimize_model(RandomForestClassifier(random_state=SEED, n_jobs=-1), 
                             rf_params, "Random Forest", X_train_final, y_train)
    best_svm = optimize_model(SVC(probability=True, random_state=SEED), 
                              svm_params, "SVM", X_train_final, y_train)
    best_gnb = GaussianNB(var_smoothing=1e-9)

    # -----------------------------------
    # D. 训练 Stacking 模型
    # -----------------------------------
    print("  --- Training Final Stacking Model ---")
    meta_learner = LogisticRegression(C=1.0, random_state=SEED)

    stacking_model = StackingClassifier(
        estimators=[
            ('xgb', best_xgb),
            ('lgbm', best_lgbm),
            ('rf', best_rf),
            ('svm', best_svm),
            ('gnb', best_gnb)
        ],
        final_estimator=meta_learner,
        cv=5, 
        n_jobs=-1,
        passthrough=False 
    )

    stacking_model.fit(X_train_final, y_train)

    # -----------------------------------
    # E. 评估当前折
    # -----------------------------------
    y_pred_prob = stacking_model.predict_proba(X_val_final)[:, 1]
    
    fpr, tpr, _ = roc_curve(y_val, y_pred_prob)
    roc_auc = auc(fpr, tpr)
    cv_metrics['auc'].append(roc_auc)
    tprs.append(np.interp(mean_fpr, fpr, tpr))
    tprs[-1][0] = 0.0
    
    precisions, recalls, thresholds = precision_recall_curve(y_val, y_pred_prob)
    with np.errstate(divide='ignore', invalid='ignore'):
        f1_scores_curve = 2 * (precisions * recalls) / (precisions + recalls)
    f1_scores_curve = np.nan_to_num(f1_scores_curve)
    
    best_idx = np.argmax(f1_scores_curve)
    best_f1 = f1_scores_curve[best_idx]
    best_thresh = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    
    y_pred_class = (y_pred_prob >= best_thresh).astype(int)
    
    acc = accuracy_score(y_val, y_pred_class)
    prec = precision_score(y_val, y_pred_class, zero_division=0)
    rec = recall_score(y_val, y_pred_class, zero_division=0)
    
    cv_metrics['acc'].append(acc)
    cv_metrics['prec'].append(prec)
    cv_metrics['recall'].append(rec)
    cv_metrics['f1'].append(best_f1)
    
    print(f"  --> Fold {fold_no} Result: AUC: {roc_auc:.4f} | F1: {best_f1:.4f} | Acc: {acc:.4f}")
    fold_no += 1

In [ ]:
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn 模块
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier, 
                              StackingClassifier)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (roc_curve, auc, precision_recall_curve, 
                             accuracy_score, precision_score, recall_score)

# 树模型框架
import xgboost as xgb
import lightgbm as lgb

# TensorFlow / Keras (用于 MLP)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Activation
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import tensorflow.keras.backend as K

# ==========================================
# 0. 环境设置
# ==========================================
sns.set_style("whitegrid")
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings("ignore")

SEED = 0
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ==========================================
# 1. 核心模型定义工厂函数
# ==========================================
def build_keras_mlp(input_dim):
    """构建 Keras MLP 模型结构"""
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(32, kernel_regularizer=l2(0.005)),
        BatchNormalization(),
        Activation('swish'),
        Dropout(0.3),
        Dense(32, kernel_regularizer=l2(0.005)),
        BatchNormalization(),
        Activation('swish'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), 
                  loss='binary_crossentropy', 
                  metrics=[tf.keras.metrics.AUC(name='auc')])
    return model

def get_models(scale_pos_weight, input_dim):
    """
    动态生成每一折的全新模型实例。
    返回一个字典，Key为模型名称，Value为模型对象。
    """
    models = {}
    
    # 1. XGBoost
    models['XGBoost'] = xgb.XGBClassifier(
        n_estimators=300, learning_rate=0.01, max_depth=3,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        scale_pos_weight=scale_pos_weight, eval_metric='auc',
        random_state=SEED, n_jobs=-1
    )
    
    # 2. LightGBM
    models['LightGBM'] = lgb.LGBMClassifier(
        n_estimators=300, learning_rate=0.01, num_leaves=15, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
        scale_pos_weight=scale_pos_weight, objective='binary',
        random_state=SEED, n_jobs=-1, verbosity=-1
    )
    
    # 3. Random Forest
    models['Random Forest'] = RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_split=5, min_samples_leaf=4,
        max_features='sqrt', class_weight='balanced',
        random_state=SEED, n_jobs=-1
    )
    
    # 4. GBDT
    models['GBDT'] = GradientBoostingClassifier(
        n_estimators=300, learning_rate=0.01, max_depth=5,
        subsample=0.8, min_samples_split=5, min_samples_leaf=5,
        random_state=SEED
    )
    
    # 5. Gaussian Naive Bayes
    models['GaussianNB'] = GaussianNB(var_smoothing=1e-8)
    
    # 6. SVM
    models['SVM'] = SVC(
        C=0.5, kernel='rbf', gamma='scale', class_weight='balanced',
        probability=True, random_state=SEED
    )
    
    # 7. MLP (Keras) - 返回结构和标识，在训练循环中专门处理
    models['MLP'] = build_keras_mlp(input_dim)
    
    # 8. Stacking (使用上述基模型)
    base_estimators = [
        ('xgb', models['XGBoost']),
        ('lgbm', models['LightGBM']),
        ('rf', models['Random Forest']),
        ('svm', models['SVM']),
        ('gnb', models['GaussianNB'])
    ]
    models['Stacking'] = StackingClassifier(
        estimators=base_estimators,
        final_estimator=LogisticRegression(class_weight='balanced', random_state=SEED),
        cv=5, n_jobs=-1, passthrough=False
    )
    
    return models

# ==========================================
# 2. 数据准备
# ==========================================
print("正在读取数据...")
df_before = pd.read_csv('全部病房患者/内部训练集/preicu.csv')
df_after = pd.read_csv('全部病房患者/内部训练集/icu.csv')

cols_stage1 = [
    'gender', 'age', 'admissionroute', 
    'temp_max', 'temp_min', 'resp_max', 'resp_min',
    'hr_max', 'hr_min', 'map_max', 'map_min',
    'tbil', 'glu', 'cre', 'plt', 'wbc', 'lym', 
    'vasopressor', 'MV'
]

cols_stage2_raw = [
    'gender', 'age', 'admissionroute',
    'temp_max', 'temp_min', 'resp_max', 'resp_min',
    'hr_max', 'hr_min', 'map_max', 'map_min',
    'glu', 'tbil', 'cre', 'spo2', 'plt',
    'wbc', 'lym', 'ph', 'lac',
    'vasopressor', 'MV'
]

binary_cols = ['gender', 'vasopressor', 'MV', 'admissionroute']
continuous_cols = [c for c in cols_stage2_raw if c not in binary_cols]

y_all = df_before['death_90d'].values
indices = np.arange(len(df_before))

assert len(df_before) == len(df_after), "两份数据文件行数不一致！"

# ==========================================
# 3. 核心 5-Fold 交叉验证循环
# ==========================================
print("\n--- 开始执行整合多模型 5-Fold 交叉验证 ---")

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# 初始化记录字典
model_names = ['XGBoost', 'LightGBM', 'Random Forest', 'GBDT', 'GaussianNB', 'SVM', 'MLP', 'Stacking']
cv_results = {name: {'auc': [], 'acc': [], 'prec': [], 'recall': [], 'f1': [], 'tprs': []} for name in model_names}

mean_fpr = np.linspace(0, 1, 100)
fold_no = 1

for train_idx, val_idx in kfold.split(indices, y_all):
    print(f"\n========== Fold {fold_no} ==========")
    K.clear_session()
    
    # -----------------------------------
    # Stage 1: Logistic Regression 生成 pre_icu_risk
    # -----------------------------------
    X_s1_train, X_s1_val = df_before.iloc[train_idx][cols_stage1], df_before.iloc[val_idx][cols_stage1]
    y_train, y_val = y_all[train_idx], y_all[val_idx]
    
    scaler_s1 = RobustScaler()
    X_s1_train_scaled = scaler_s1.fit_transform(X_s1_train)
    X_s1_val_scaled = scaler_s1.transform(X_s1_val)
    
    lr_s1 = LogisticRegression(class_weight='balanced', solver='liblinear', random_state=SEED)
    lr_s1.fit(X_s1_train_scaled, y_train)
    
    risk_train = lr_s1.predict_proba(X_s1_train_scaled)[:, 1]
    risk_val = lr_s1.predict_proba(X_s1_val_scaled)[:, 1]
    
    # -----------------------------------
    # Stage 2: 特征整合与预处理
    # -----------------------------------
    X_s2_train_raw, X_s2_val_raw = df_after.iloc[train_idx][cols_stage2_raw].copy(), df_after.iloc[val_idx][cols_stage2_raw].copy()
    X_s2_train_raw['pre_icu_risk'], X_s2_val_raw['pre_icu_risk'] = risk_train, risk_val
    
    current_cont_cols = continuous_cols + ['pre_icu_risk']
    
    # 统一管道处理缺失值和归一化
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler())
    ])
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent'))
    ])
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('cont', numeric_transformer, current_cont_cols),
            ('cat', categorical_transformer, binary_cols)
        ],
        verbose_feature_names_out=False
    )
    
    X_train_final = preprocessor.fit_transform(X_s2_train_raw).astype(np.float32)
    X_val_final = preprocessor.transform(X_s2_val_raw).astype(np.float32)
    
    # -----------------------------------
    # 模型训练与评估遍历
    # -----------------------------------
    pos_count, neg_count = np.sum(y_train == 1), np.sum(y_train == 0)
    scale_pos_weight = neg_count / pos_count if pos_count > 0 else 1.0
    
    # 获取此 Fold 的全新模型字典
    models_dict = get_models(scale_pos_weight, input_dim=X_train_final.shape[1])
    
    for model_name in model_names:
        print(f"  Training {model_name}...")
        model = models_dict[model_name]
        
        # 针对不同模型的 Fit 和 Predict_Proba 处理
        if model_name == 'MLP':
            # 计算样本权重
            class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
            cw_dict = dict(enumerate(class_weights))
            
            callbacks = [
                EarlyStopping(monitor='val_auc', patience=20, restore_best_weights=True, mode='max', verbose=0),
                ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=10, min_lr=1e-5, mode='max', verbose=0)
            ]
            model.fit(X_train_final, y_train, validation_data=(X_val_final, y_val),
                      epochs=150, batch_size=32, callbacks=callbacks, class_weight=cw_dict, verbose=0)
            y_pred_prob = model.predict(X_val_final, verbose=0).ravel()
            
        elif model_name == 'XGBoost':
            # 带有 Early Stopping 的树模型
            model.fit(X_train_final, y_train, eval_set=[(X_val_final, y_val)], verbose=False)
            y_pred_prob = model.predict_proba(X_val_final)[:, 1]
        
        elif model_name == 'LightGBM':
            model.fit(X_train_final, y_train, eval_set=[(X_val_final, y_all[val_idx])], eval_metric='auc')
            y_pred_prob = model.predict_proba(X_val_final)[:, 1]

        else:
            # 标准 Scikit-Learn 模型 (RF, GBDT, GNB, SVM, Stacking)
            model.fit(X_train_final, y_train)
            y_pred_prob = model.predict_proba(X_val_final)[:, 1]
            
        # --- 计算指标 ---
        # 1. ROC/AUC
        fpr, tpr, _ = roc_curve(y_val, y_pred_prob)
        roc_auc = auc(fpr, tpr)
        
        # 记录 TPR 用于平均 ROC 曲线
        interp_tpr = np.interp(mean_fpr, fpr, tpr)
        interp_tpr[0] = 0.0
        cv_results[model_name]['tprs'].append(interp_tpr)
        cv_results[model_name]['auc'].append(roc_auc)
        
        # 2. 寻找最佳阈值并计算其他指标
        precisions, recalls, thresholds = precision_recall_curve(y_val, y_pred_prob)
        with np.errstate(divide='ignore', invalid='ignore'):
            f1_scores = np.nan_to_num(2 * (precisions * recalls) / (precisions + recalls))
            
        best_idx = np.argmax(f1_scores)
        best_f1 = f1_scores[best_idx]
        best_thresh = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
        
        y_pred_class = (y_pred_prob >= best_thresh).astype(int)
        acc = accuracy_score(y_val, y_pred_class)
        prec = precision_score(y_val, y_pred_class, zero_division=0)
        rec = recall_score(y_val, y_pred_class, zero_division=0)
        
        cv_results[model_name]['f1'].append(best_f1)
        cv_results[model_name]['acc'].append(acc)
        cv_results[model_name]['prec'].append(prec)
        cv_results[model_name]['recall'].append(rec)

    fold_no += 1

# ==========================================
# 4. 指标汇总与打印
# ==========================================
print("\n" + "="*80)
print(f"{'Model':<15} | {'AUC (Mean ± Std)':<18} | {'F1 (Mean ± Std)':<18} | {'Acc (Mean ± Std)':<18} | {'Prec (Mean ± Std)':<18} | {'Recall (Mean ± Std)':<18}")
print("="*80)

final_stats = {}
for name in model_names:
    stats = {}
    for metric in ['auc', 'f1', 'acc', 'prec', 'recall']:
        mean_val = np.mean(cv_results[name][metric])
        std_val = np.std(cv_results[name][metric])
        stats[f'{metric}_mean'] = mean_val
        stats[f'{metric}_std'] = std_val
    final_stats[name] = stats
    
    print(f"{name:<15} | {stats['auc_mean']:.3f} ± {stats['auc_std']:.3f} | {stats['f1_mean']:.3f} ± {stats['f1_std']:.3f} | {stats['acc_mean']:.3f} ± {stats['acc_std']:.3f} | {stats['prec_mean']:.3f} ± {stats['prec_std']:.3f} | {stats['recall_mean']:.3f} ± {stats['recall_std']:.3f}")

print("="*80)

# ==========================================
# 5. 一张图绘制所有模型的 平均 ROC 曲线
# ==========================================
plt.figure(figsize=(10, 9), dpi=150)

# 定义各个模型的绘图颜色 (选用醒目的调色板)
colors = {
    'XGBoost': '#1f77b4',       # 蓝色
    'LightGBM': '#ff7f0e',      # 橙色
    'Random Forest': '#2ca02c', # 绿色
    'GBDT': '#d62728',          # 红色
    'GaussianNB': '#9467bd',    # 紫色
    'SVM': '#8c564b',           # 棕色
    'MLP': '#e377c2',           # 粉色
    'Stacking': '#17becf'       # 青色
}

for name in model_names:
    mean_tpr = np.mean(cv_results[name]['tprs'], axis=0)
    mean_tpr[-1] = 1.0
    mean_auc = final_stats[name]['auc_mean']
    std_auc = final_stats[name]['auc_std']
    
    # 画平均曲线
    plt.plot(mean_fpr, mean_tpr, color=colors[name], lw=2.5, alpha=0.9,
             label=f'{name} (AUC = {mean_auc:.3f} ± {std_auc:.3f})')
    
    # (可选) 若想查看方差带，可取消以下注释。但多个模型叠加方差带会让图显得过于杂乱
    # std_tpr = np.std(cv_results[name]['tprs'], axis=0)
    # tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
    # tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
    # plt.fill_between(mean_fpr, tprs_lower, tprs_upper, color=colors[name], alpha=0.1)

# 对角线（随机猜测）
plt.plot([0, 1], [0, 1], linestyle='--', lw=2, color='gray', alpha=0.8, label='Chance')

# 图表装饰
plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.02])
plt.xlabel('False Positive Rate', fontsize=14, fontweight='bold')
plt.ylabel('True Positive Rate', fontsize=14, fontweight='bold')
plt.title('Comparison of Mean ROC Curves (Two-Stage Models)', fontsize=16, fontweight='bold', pad=20)
plt.legend(loc="lower right", fontsize=11, frameon=True, fancybox=True, edgecolor='lightgray')
plt.grid(True, linestyle=':', linewidth=0.6, alpha=0.8)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.utils import resample
from sklearn.metrics import (roc_curve, auc, precision_recall_curve, 
                             accuracy_score, precision_score, recall_score, f1_score)
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

# ==========================================
# 0. 环境设置
# ==========================================
sns.set_style("whitegrid")
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings("ignore")

SEED = 0
np.random.seed(SEED)

# ==========================================
# 1. 数据准备
# ==========================================
print("正在读取数据...")

# --- A. 读取内部训练和验证集 ---
df_before_int = pd.read_csv('全部病房患者/内部训练集/preicu.csv')
df_after_int = pd.read_csv('全部病房患者/内部训练集/icu.csv')

# --- B. 读取外部测试集 ---
df_before_ext = pd.read_csv('全部病房患者/外部测试集/preicu.csv')
df_after_ext = pd.read_csv('全部病房患者/外部测试集/icu.csv')

print(f"训练/内验总样本数: {len(df_before_int)}")
print(f"外部测试总样本数: {len(df_before_ext)}")

# --- C. 定义特征 ---
cols_stage1 = [
    'gender', 'age', 'admissionroute', 
    'temp_max', 'temp_min', 'resp_max', 'resp_min',
    'hr_max', 'hr_min', 'map_max', 'map_min',
    'tbil', 'glu', 'cre', 'plt', 'wbc', 'lym', 
    'vasopressor', 'MV'
]

cols_stage2_raw = [
    'gender', 'age',
    'temp_max', 'temp_min', 'resp_max', 'resp_min',
    'hr_max', 'hr_min', 'map_max', 'map_min',
    'glu', 'tbil', 'cre', 'spo2', 'plt',
    'wbc', 'lym', 'ph', 'lac',
    'vasopressor', 'MV'
]

binary_cols = ['gender', 'vasopressor', 'MV', 'admissionroute']
continuous_cols = [c for c in cols_stage2_raw if c not in ['gender', 'vasopressor', 'MV']] 

y_int = df_before_int['death_90d'].values
y_ext = df_before_ext['death_90d'].values

# ==========================================
# 2. 数据集划分 (Train / Internal Val)
# ==========================================
indices_int = np.arange(len(df_before_int))
train_idx, val_idx = train_test_split(
    indices_int, test_size=0.2, stratify=y_int, random_state=SEED
)
print(f"\n划分完成: 训练集 {len(train_idx)} 人, 内部验证集 {len(val_idx)} 人")

# ==========================================
# 3. 第一阶段：生成 Pre-ICU Risk Score
# ==========================================
print("\n--- Stage 1: Logistic Regression for Risk Score ---")
X_s1_train = df_before_int.iloc[train_idx][cols_stage1].reset_index(drop=True)
y_train = y_int[train_idx]
X_s1_val = df_before_int.iloc[val_idx][cols_stage1].reset_index(drop=True)
X_s1_ext = df_before_ext[cols_stage1].reset_index(drop=True)

scaler_s1 = RobustScaler()
X_s1_train_scaled = scaler_s1.fit_transform(X_s1_train)
X_s1_val_scaled = scaler_s1.transform(X_s1_val)
X_s1_ext_scaled = scaler_s1.transform(X_s1_ext)

lr_s1 = LogisticRegression(class_weight='balanced', solver='liblinear', random_state=SEED)
lr_s1.fit(X_s1_train_scaled, y_train)

risk_train = lr_s1.predict_proba(X_s1_train_scaled)[:, 1]
risk_val = lr_s1.predict_proba(X_s1_val_scaled)[:, 1]
risk_ext = lr_s1.predict_proba(X_s1_ext_scaled)[:, 1]

# ==========================================
# 4. 第二阶段：特征预处理
# ==========================================
print("\n--- Stage 2: Data Preprocessing ---")
X_s2_train_raw = df_after_int.iloc[train_idx][cols_stage2_raw].reset_index(drop=True)
X_s2_val_raw = df_after_int.iloc[val_idx][cols_stage2_raw].reset_index(drop=True)
X_s2_ext_raw = df_after_ext[cols_stage2_raw].reset_index(drop=True)

X_s2_train_raw['pre_icu_risk'] = risk_train
X_s2_val_raw['pre_icu_risk'] = risk_val
X_s2_ext_raw['pre_icu_risk'] = risk_ext

cols_to_scale = continuous_cols + ['pre_icu_risk']
cols_passthrough = [c for c in cols_stage2_raw if c in binary_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('cont', RobustScaler(), cols_to_scale),
        ('cat', 'passthrough', cols_passthrough)
    ],
    verbose_feature_names_out=False
)
preprocessor.set_output(transform='pandas')

X_train_final = preprocessor.fit_transform(X_s2_train_raw)
X_val_final = preprocessor.transform(X_s2_val_raw)
X_ext_final = preprocessor.transform(X_s2_ext_raw)

y_val = y_int[val_idx]

# ==========================================
# 5. 超参数调优 (RandomizedSearchCV)
# ==========================================
print("\n--- Hyperparameter Optimization (RandomizedSearch) ---")

pos_count = np.sum(y_train == 1)
neg_count = np.sum(y_train == 0)
scale_pos_weight = neg_count / pos_count if pos_count > 0 else 1.0

# 参数空间定义
xgb_params = {
    'n_estimators': [300, 500, 700],
    'learning_rate': [0.01],
    'max_depth': [3, 4, 5],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'min_child_weight': [2, 3, 4],
    'scale_pos_weight': [scale_pos_weight]
}

lgb_params = {
    'n_estimators': [300, 500, 700],
    'num_leaves': [15, 31],
    'learning_rate': [0.01],
    'max_depth': [4, 5, 6],
    'subsample': [0.7, 0.8, 0.9],
    'min_child_samples': [10, 20],
    'scale_pos_weight': [scale_pos_weight]
}

rf_params = {
    'n_estimators': [300, 500, 700],
    'max_depth': [5, 10],
    'min_samples_split': [3, 4, 5],
    'min_samples_leaf': [2, 3, 4],
    'max_features': ['sqrt'],
    'class_weight': ['balanced']
}

svm_params = {
    'C': [0.5, 1.0, 2.0],
    'kernel': ['rbf', 'linear'],
    'gamma': ['scale', 'auto']
}

def optimize_model(estimator, param_dist, name):
    print(f"正在优化 {name} ...")
    search = RandomizedSearchCV(
        estimator, param_distributions=param_dist, 
        n_iter=100,
        scoring='roc_auc', cv=5, n_jobs=-1, random_state=SEED, verbose=0
    )
    search.fit(X_train_final, y_train)
    return search.best_estimator_

best_xgb = optimize_model(xgb.XGBClassifier(eval_metric='auc', random_state=SEED, n_jobs=1), 
                          xgb_params, "XGBoost")
best_lgbm = optimize_model(lgb.LGBMClassifier(objective='binary', metric='auc', verbosity=-1, random_state=SEED, n_jobs=1), 
                           lgb_params, "LightGBM")
best_rf = optimize_model(RandomForestClassifier(random_state=SEED, n_jobs=-1), 
                         rf_params, "Random Forest")
best_svm = optimize_model(SVC(probability=True, random_state=SEED), 
                           svm_params, "SVM")
best_gnb = GaussianNB(var_smoothing=1e-9)

# ==========================================
# 6. 训练 Stacking 模型
# ==========================================
print("\n--- Training Final Stacking Model ---")

meta_learner = LogisticRegression(class_weight='balanced', random_state=SEED)

stacking_model = StackingClassifier(
    estimators=[
        ('xgb', best_xgb),
        ('lgbm', best_lgbm),
        ('rf', best_rf),
        ('svm', best_svm),
        ('gnb', best_gnb)
    ],
    final_estimator=meta_learner,
    cv=5, 
    n_jobs=-1,
    passthrough=False 
)

stacking_model.fit(X_train_final, y_train)
print("模型训练完成！")

# ==========================================
# 7. Bootstrap 95% CI 评估 
# ==========================================
print("\n正在计算最佳分类阈值 (基于 Internal Val F1)...")
val_probs = stacking_model.predict_proba(X_val_final)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, val_probs)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_scores)
BEST_THRESHOLD = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
print(f"最佳阈值: {BEST_THRESHOLD:.4f}")

def bootstrap_metrics_ci_with_roc(model, X, y, threshold, n_bootstraps=1000, dataset_name="Dataset"):
    print(f"\n正在进行 Bootstrap 评估 ({dataset_name})...")
    
    y_prob_full = model.predict_proba(X)[:, 1]
    
    boot_stats = {
        'AUC': [], 'Accuracy': [], 'Precision': [], 'Recall': [], 'F1': []
    }

    mean_fpr = np.linspace(0, 1, 100)
    tprs = []
    
    for i in range(n_bootstraps):
        indices = resample(np.arange(len(y)), random_state=i)
        y_true_boot = y[indices]
        y_prob_boot = y_prob_full[indices]
        
        if len(np.unique(y_true_boot)) < 2:
            continue
            
        fpr, tpr, _ = roc_curve(y_true_boot, y_prob_boot)
        boot_stats['AUC'].append(auc(fpr, tpr))
        
        # 收集插值后的 TPR
        interp_tpr = np.interp(mean_fpr, fpr, tpr)
        interp_tpr[0] = 0.0
        tprs.append(interp_tpr)
        
        y_pred_boot = (y_prob_boot >= threshold).astype(int)
        
        boot_stats['Accuracy'].append(accuracy_score(y_true_boot, y_pred_boot))
        boot_stats['Precision'].append(precision_score(y_true_boot, y_pred_boot, zero_division=0))
        boot_stats['Recall'].append(recall_score(y_true_boot, y_pred_boot, zero_division=0))
        boot_stats['F1'].append(f1_score(y_true_boot, y_pred_boot, zero_division=0))

    print(f">>> {dataset_name} Results (Mean [95% CI]):")
    results_str = {}
    
    for metric, values in boot_stats.items():
        mean_val = np.mean(values)
        ci_lower = np.percentile(values, 2.5)
        ci_upper = np.percentile(values, 97.5)
        
        res_str = f"{mean_val:.3f} ({ci_lower:.3f}-{ci_upper:.3f})"
        results_str[metric] = res_str
        print(f"{metric:<10}: {res_str}")
        
    # 计算 ROC 曲线的均值和上下界
    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[-1] = 1.0
    tpr_lower = np.percentile(tprs, 2.5, axis=0)
    tpr_upper = np.percentile(tprs, 97.5, axis=0)
        
    return boot_stats, results_str, mean_fpr, mean_tpr, tpr_lower, tpr_upper

# 运行评估并获取作图所需数据
stats_val, str_val, fpr_val, tpr_val, tpr_val_lower, tpr_val_upper = bootstrap_metrics_ci_with_roc(
    stacking_model, X_val_final, y_val, BEST_THRESHOLD, dataset_name="Internal Validation"
)

stats_ext, str_ext, fpr_ext, tpr_ext, tpr_ext_lower, tpr_ext_upper = bootstrap_metrics_ci_with_roc(
    stacking_model, X_ext_final, y_ext, BEST_THRESHOLD, dataset_name="External Test"
)

# ==========================================
# 8. 可视化 ROC
# ==========================================
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['axes.linewidth'] = 1.5

fig, ax = plt.subplots(figsize=(7, 7), dpi=300)

color_val = '#00468B'
color_ext = '#ED0000'

ax.plot(fpr_val, tpr_val, color=color_val, lw=2.5, 
        label=f'Internal Val (AUC = {str_val["AUC"]})')
ax.fill_between(fpr_val, tpr_val_lower, tpr_val_upper, color=color_val, alpha=0.15, linewidth=0)

ax.plot(fpr_ext, tpr_ext, color=color_ext, lw=2.5, linestyle='-',
        label=f'External Test (AUC = {str_ext["AUC"]})')
ax.fill_between(fpr_ext, tpr_ext_lower, tpr_ext_upper, color=color_ext, alpha=0.15, linewidth=0)
ax.plot([0, 1], [0, 1], linestyle='--', lw=2, color='gray', alpha=0.8)
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
ax.set_xlabel('1 - Specificity (False Positive Rate)', fontsize=13, fontweight='medium')
ax.set_ylabel('Sensitivity (True Positive Rate)', fontsize=13, fontweight='medium')
ax.set_title('Receiver Operating Characteristic (ROC) Analysis', 
             fontsize=15, fontweight='bold', pad=15)
ax.legend(loc="lower right", fontsize=11, frameon=False)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import copy
import matplotlib.pyplot as plt
from scipy.spatial import distance
from scipy.optimize import linear_sum_assignment  # 引入线性分配算法
from sklearn.linear_model import LogisticRegression

# ==========================================
# 核心配置区域
# ==========================================
# 判定平衡的阈值
SMD_THRESHOLD = 0.2

# 关键：定义用于计算“马氏距离”的变量
# 这些应该是最能代表“病情严重程度”的变量
SEVERITY_VARS = [
    'MV', 'admissionroute', 'vasopressor', 'age'
]

# ==========================================
# 工具函数
# ==========================================

def calculate_propensity_scores(X, treatment_col, covariate_list):
    """计算倾向性评分 (PS) 和 Logit PS"""
    # class_weight='balanced' 处理不平衡数据
    model = LogisticRegression(solver='liblinear', class_weight='balanced', random_state=42)
    model.fit(X[covariate_list], X[treatment_col])
    ps_score = model.predict_proba(X[covariate_list])[:, 1]
    
    X = X.copy()
    X['ps'] = ps_score
    # 防止 log(0) 或 log(1)
    X['ps'] = X['ps'].clip(1e-6, 1-1e-6)
    # 计算 Logit PS
    X['ps_logit'] = np.log(X['ps'] / (1 - X['ps']))
    
    return X

def get_mahalanobis_params(X, vars_list):
    """预计算马氏距离需要的协方差矩阵的逆矩阵"""
    cov_matrix = np.cov(X[vars_list].T)
    try:
        inv_cov_matrix = np.linalg.inv(cov_matrix)
    except np.linalg.LinAlgError:
        inv_cov_matrix = np.linalg.pinv(cov_matrix)
    return inv_cov_matrix

def check_balance(X, treatment_col, covariates):
    """计算标准化均值差 (SMD)"""
    treated = X[X[treatment_col] == 1][covariates]
    control = X[X[treatment_col] == 0][covariates]
    
    if len(treated) == 0 or len(control) == 0:
        return pd.Series(index=covariates, dtype=float)

    mean_t = treated.mean()
    mean_c = control.mean()
    std_t = treated.std()
    std_c = control.std()
    
    pooled_std = np.sqrt((std_t**2 + std_c**2) / 2)
    pooled_std[pooled_std == 0] = 1e-9 
    
    smd = (mean_t - mean_c).abs() / pooled_std
    return smd

# ==========================================
# 核心：全局最优匹配算法
# ==========================================

def perform_global_optimal_matching(df, treatment_col, covariates, severity_vars, caliper_scale=0.2):
    """
    执行全局最优匹配 (Global Optimal Matching)：
    1. 计算处理组和对照组之间的全量距离矩阵（Cost Matrix）。
    2. 距离 = 马氏距离 (Severity Vars) + 惩罚项 (如果 PS Logit 差值超过卡尺)。
    3. 使用匈牙利算法 (Linear Sum Assignment) 最小化总距离。
    """
    # 1. 计算 PS
    df_data = calculate_propensity_scores(df, treatment_col, covariates)
    
    # 2. 分离组别
    treated_df = df_data[df_data[treatment_col] == 1]
    control_df = df_data[df_data[treatment_col] == 0]
    
    n_treated = len(treated_df)
    n_control = len(control_df)
    
    print(f"开始全局最优匹配... 处理组: {n_treated}, 对照组: {n_control}")
    
    if n_control == 0 or n_treated == 0:
        print("某一组为空，无法匹配")
        return pd.DataFrame()

    # 3. 准备参数
    # 卡尺计算
    std_logit = df_data['ps_logit'].std()
    caliper = caliper_scale * std_logit
    print(f"设定卡尺 (Logit Scale): {caliper:.4f}")
    
    # 马氏距离逆矩阵
    VI = get_mahalanobis_params(df_data, severity_vars)
    
    # 4. 构建代价矩阵 (Cost Matrix)
    
    # (A) 计算 Logit PS 的绝对差值矩阵
    # 利用广播机制： (N, 1) - (1, M) = (N, M)
    t_logits = treated_df['ps_logit'].values.reshape(-1, 1)
    c_logits = control_df['ps_logit'].values.reshape(1, -1)
    dist_logit = np.abs(t_logits - c_logits)
    
    # (B) 计算 Severity Vars 的马氏距离矩阵
    t_feats = treated_df[severity_vars].values
    c_feats = control_df[severity_vars].values
    dist_mahal = distance.cdist(t_feats, c_feats, metric='mahalanobis', VI=VI)
    
    # (C) 融合距离与卡尺惩罚
    # 逻辑：基础代价是马氏距离。
    # 如果 Logit 距离超过卡尺，给予一个极大的惩罚值 (PENALTY)，使其在最优解中被弃用。
    # 注意：不能用 np.inf，因为线性规划求解器不支持 inf，要用一个足够大的数。
    PENALTY = 1e6 
    cost_matrix = dist_mahal.copy()
    
    # 将超过卡尺的配对距离设为超大值
    out_of_caliper_mask = dist_logit > caliper
    cost_matrix[out_of_caliper_mask] = PENALTY
    
    print("代价矩阵构建完成，正在求解最优分配问题 (Hungarian Algorithm)...")
    
    # 5. 求解二部图最小权匹配 (匈牙利算法)
    # row_ind 对应 treated_df 的行下标 (0 到 n_treated-1)
    # col_ind 对应 control_df 的行下标 (0 到 n_control-1)
    # scipy 会自动处理矩形矩阵，尽可能多地匹配
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    
    # 6. 提取有效匹配
    valid_matches_t = []
    valid_matches_c = []
    
    for r, c in zip(row_ind, col_ind):
        # 检查是否真的匹配成功（即距离没有因为超过卡尺而被设为 PENALTY）
        if cost_matrix[r, c] < PENALTY:
            valid_matches_t.append(treated_df.index[r])
            valid_matches_c.append(control_df.index[c])
            
    matched_t_df = df_data.loc[valid_matches_t]
    matched_c_df = df_data.loc[valid_matches_c]
    
    # 合并
    df_matched = pd.concat([matched_t_df, matched_c_df])
    
    print(f"匹配结束。")
    print(f"处理组原始: {n_treated} -> 匹配成功: {len(valid_matches_t)}")
    print(f"匹配率: {len(valid_matches_t)/n_treated*100:.2f}%")
    
    return df_matched

# ==========================================
# 主执行流程
# ==========================================

# 1. 读取数据
filename = '全部病房患者/内部训练集/preicu.csv'
df_raw = pd.read_csv(filename)
df = df_raw.copy()
df['T'] = df['lac'].notnull().astype(int)
print("数据加载成功。")

# 2. 定义变量
covariates = [
    'gender', 'age', 'admissionroute', 
    'temp_max', 'temp_min', 'resp_max', 'resp_min', 
    'hr_max', 'hr_min', 'map_max', 'map_min', 
    'tbil', 'glu', 'cre', 'plt', 'wbc', 'lym', 
    'vasopressor', 'MV'
]

# 确保数据无缺失
df_clean = df.dropna(subset=covariates).reset_index(drop=True)

# 3. 检查匹配前平衡性
print("\n--- 匹配前平衡性 ---")
pre_smd = check_balance(df_clean, 'T', covariates)
print(f"SMD > {SMD_THRESHOLD} 的变量数: {(pre_smd > SMD_THRESHOLD).sum()}")
print(pre_smd)

# 4. 执行全局最优匹配
# 使用较紧的 caliper (例如 0.1 或 0.05) 以获得高质量匹配
df_matched = perform_global_optimal_matching(
    df_clean, 
    'T', 
    covariates, 
    severity_vars=SEVERITY_VARS, 
    caliper_scale=0.1
)

# 5. 检查匹配后平衡性
if not df_matched.empty:
    print("\n--- 匹配后平衡性 ---")
    post_smd = check_balance(df_matched, 'T', covariates)
    print(f"SMD > {SMD_THRESHOLD} 的变量数: {(post_smd > SMD_THRESHOLD).sum()}")
    print("主要重症指标的 SMD:")
    print(post_smd)
    
    # 6. 保存
    df_matched.to_csv('全部病房患者/内部训练集/rmyy_matched_optimal.csv', index=False)
    print("\n文件已保存至 rmyy_matched_optimal.csv")

    # 7. 可视化
    plt.figure(figsize=(10, 6))
    plt.hist(df_matched[df_matched['T']==1]['ps'], bins=30, alpha=0.5, label='Treated (Lac)', density=True)
    plt.hist(df_matched[df_matched['T']==0]['ps'], bins=30, alpha=0.5, label='Control (No Lac)', density=True)
    plt.title('Propensity Score Distribution (Global Optimal Matching)')
    plt.xlabel('Propensity Score')
    plt.legend()
    plt.show()
else:
    print("匹配结果为空，可能是卡尺过紧或无重叠区域。")

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier, 
                              StackingClassifier)
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.utils import resample
from sklearn.metrics import (roc_curve, auc, precision_recall_curve, 
                             accuracy_score, precision_score, recall_score, f1_score, 
                             roc_auc_score, brier_score_loss) # 新增 brier_score_loss
from sklearn.calibration import calibration_curve
from scipy.stats import norm # 用于计算P值
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

# ==========================================
# 0. 全局设置
# ==========================================
sns.set_style("whitegrid")
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings("ignore")

SEED = 0
np.random.seed(SEED)

# ==========================================
# 1. 数据加载与合并
# ==========================================
def load_and_merge_data(matched_file, icu_file):
    if not os.path.exists(matched_file) or not os.path.exists(icu_file):
        raise FileNotFoundError(f"文件未找到: {matched_file} 或 {icu_file}")
    
    df_matched = pd.read_csv(matched_file)
    df_icu = pd.read_csv(icu_file)
    
    valid_ids = df_matched['pat_id'].unique()
    df_icu = df_icu[df_icu['pat_id'].isin(valid_ids)].copy()
    
    icu_cols = [c for c in df_icu.columns if c != 'pat_id']
    rename_dict = {c: f"{c}_s2" for c in icu_cols}
    df_icu = df_icu.rename(columns=rename_dict)
    
    df_merged = pd.merge(df_matched, df_icu, on='pat_id', how='inner')
    return df_merged

print(">>> Loading Development Data ...")
df_dev_all = load_and_merge_data('全部病房患者/内部训练集/rmyy_matched_optimal.csv', '全部病房患者/内部训练集/icu.csv')

print(">>> Loading External Test Data (23-25)...")
df_test_all = load_and_merge_data('全部病房患者/外部测试集/rmyy_matched_optimal.csv', '全部病房患者/外部测试集/icu.csv')

print(f"Dev Set Total: {len(df_dev_all)} | Ext Test Set Total: {len(df_test_all)}")

# ==========================================
# 2. 特征定义
# ==========================================
target_col = 'death_90d'

cols_stage1_base = [
    'gender', 'age', 'admissionroute', 
    'temp_max', 'temp_min', 'resp_max', 'resp_min',
    'hr_max', 'hr_min', 'map_max', 'map_min',
    'tbil', 'glu', 'cre', 'plt', 'wbc', 'lym', 
    'vasopressor', 'MV'
]

cols_stage2_base = [
    'gender_s2', 'age_s2', 'admissionroute_s2',
    'temp_max_s2', 'temp_min_s2', 'resp_max_s2', 'resp_min_s2',
    'hr_max_s2', 'hr_min_s2', 'map_max_s2', 'map_min_s2',
    'glu_s2', 'tbil_s2', 'cre_s2', 'spo2_s2', 'plt_s2',
    'wbc_s2', 'lym_s2', 'ph_s2', 'lac_s2', 
    'vasopressor_s2', 'MV_s2'
]

binary_cols_s1 = ['gender', 'vasopressor', 'MV', 'admissionroute']
binary_cols_s2 = ['gender_s2', 'vasopressor_s2', 'MV_s2', 'admissionroute_s2']

# ==========================================
# 3. 辅助函数：Bootstrap & 统计检验
# ==========================================
def get_bootstrap_stats(model, X, y, threshold=0.5, n_bootstraps=200):
    """
    执行 Bootstrap 重采样，返回格式化字符串和原始数据列表
    """
    y_prob_full = model.predict_proba(X)[:, 1]
    
    # 存储原始数据用于统计检验
    raw_stats = {
        'AUC': [], 'Accuracy': [], 'Precision': [], 'Recall': [], 'F1': []
    }
    
    for i in range(n_bootstraps):
        indices = resample(np.arange(len(y)), random_state=i)
        if len(np.unique(y[indices])) < 2: continue
        
        y_true = y[indices]
        y_prob = y_prob_full[indices]
        y_pred = (y_prob >= threshold).astype(int)
        
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        raw_stats['AUC'].append(auc(fpr, tpr))
        raw_stats['Accuracy'].append(accuracy_score(y_true, y_pred))
        raw_stats['Precision'].append(precision_score(y_true, y_pred, zero_division=0))
        raw_stats['Recall'].append(recall_score(y_true, y_pred, zero_division=0))
        raw_stats['F1'].append(f1_score(y_true, y_pred, zero_division=0))
        
    # 生成显示用的字符串: Mean (95% CI)
    str_stats = {}
    for k, v in raw_stats.items():
        mean_v = np.mean(v)
        ci_lower = np.percentile(v, 2.5)
        ci_upper = np.percentile(v, 97.5)
        str_stats[k] = f"{mean_v:.3f} ({ci_lower:.3f}-{ci_upper:.3f})"
    
    return str_stats, raw_stats, y_prob_full

def calculate_p_value(dist_a, dist_b):
    """
    比较两个独立的 Bootstrap 分布，计算 P 值 (Z-test)
    H0: Mean_A == Mean_B
    H1: Mean_A != Mean_B (双尾检验)
    """
    mean_a, std_a = np.mean(dist_a), np.std(dist_a, ddof=1)
    mean_b, std_b = np.mean(dist_b), np.std(dist_b, ddof=1)
    
    # 计算 Z 分数 (独立样本)
    # Z = (Mean_B - Mean_A) / sqrt(SE_A^2 + SE_B^2)
    # 这里 SE 直接用 Bootstrap 分布的标准差近似
    z_score = (mean_b - mean_a) / np.sqrt(std_a**2 + std_b**2)
    
    # 计算双尾 P 值
    p_value = 2 * (1 - norm.cdf(abs(z_score)))
    
    return p_value, z_score

def calculate_net_benefit(y_true, y_prob, thresholds):
    """
    计算 DCA 的 Net Benefit
    """
    net_benefits = []
    n = len(y_true)
    for thresh in thresholds:
        # 预测为正的样本
        pred = (y_prob >= thresh).astype(int)
        tp = np.sum((pred == 1) & (y_true == 1))
        fp = np.sum((pred == 1) & (y_true == 0))
        
        # Net Benefit 公式
        # NB = (TP / N) - (FP / N) * (Pt / (1 - Pt))
        if thresh == 1.0:
            nb = 0
        else:
            nb = (tp / n) - (fp / n) * (thresh / (1 - thresh))
        net_benefits.append(nb)
    return np.array(net_benefits)

# ==========================================
# 4. 核心训练管道
# ==========================================
def run_two_stage_pipeline(df_dev, df_test, group_name, use_lac_in_s1):
    print(f"\n{'='*15} Processing: {group_name} {'='*15}")
    
    # --- 1. 特征选择 ---
    current_cols_s1 = cols_stage1_base.copy()
    if use_lac_in_s1:
        current_cols_s1.append('lac')
        print("  -> Stage 1: Included 'lac'")
    else:
        print("  -> Stage 1: No 'lac'")

    current_cols_s2 = cols_stage2_base.copy()
    if 'lac_s2' not in current_cols_s2:
        current_cols_s2.append('lac_s2')
    print("  -> Stage 2: Included 'lac_s2' (Default)")

    # 移除缺失列
    missing_s2 = [c for c in current_cols_s2 if c not in df_dev.columns]
    if missing_s2:
        current_cols_s2 = [c for c in current_cols_s2 if c not in missing_s2]

    # --- 2. 数据划分 ---
    X_dev = df_dev
    y_dev = df_dev[target_col].values
    
    train_idx, val_idx = train_test_split(
        np.arange(len(df_dev)), test_size=0.2, stratify=y_dev, random_state=SEED
    )
    
    X_ext = df_test
    y_ext = df_test[target_col].values
    
    # --- 3. Stage 1: Risk Score ---
    print("  [Step 1] Training Pre-ICU Risk Score...")
    X_s1_train = X_dev.iloc[train_idx][current_cols_s1].reset_index(drop=True)
    y_train = y_dev[train_idx]
    X_s1_val = X_dev.iloc[val_idx][current_cols_s1].reset_index(drop=True)
    X_s1_ext = X_ext[current_cols_s1].reset_index(drop=True)
    
    # 管道预处理
    cont_cols_s1 = [c for c in current_cols_s1 if c not in binary_cols_s1]
    bin_cols_s1_curr = [c for c in current_cols_s1 if c in binary_cols_s1]

    num_pipe_s1 = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', RobustScaler())])
    cat_pipe_s1 = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('passthrough', 'passthrough')])

    preprocessor_s1 = ColumnTransformer([
        ('num', num_pipe_s1, cont_cols_s1),
        ('cat', cat_pipe_s1, bin_cols_s1_curr)
    ])
    preprocessor_s1.set_output(transform='pandas')

    X_s1_train_sc = preprocessor_s1.fit_transform(X_s1_train)
    X_s1_val_sc = preprocessor_s1.transform(X_s1_val)
    X_s1_ext_sc = preprocessor_s1.transform(X_s1_ext)
    
    lr_s1 = LogisticRegression(class_weight='balanced', solver='liblinear', random_state=SEED)
    lr_s1.fit(X_s1_train_sc, y_train)
    
    risk_train = lr_s1.predict_proba(X_s1_train_sc)[:, 1]
    risk_val = lr_s1.predict_proba(X_s1_val_sc)[:, 1]
    risk_ext = lr_s1.predict_proba(X_s1_ext_sc)[:, 1]
    
    # --- 4. Stage 2: Stacking Prep ---
    print("  [Step 2] Preparing Stacking Data...")
    X_s2_train = X_dev.iloc[train_idx][current_cols_s2].reset_index(drop=True)
    X_s2_val = X_dev.iloc[val_idx][current_cols_s2].reset_index(drop=True)
    X_s2_ext = X_ext[current_cols_s2].reset_index(drop=True)
    
    X_s2_train['pre_icu_risk'] = risk_train
    X_s2_val['pre_icu_risk'] = risk_val
    X_s2_ext['pre_icu_risk'] = risk_ext
    
    cont_cols_s2 = [c for c in current_cols_s2 if c not in binary_cols_s2] + ['pre_icu_risk']
    bin_cols_s2_curr = [c for c in current_cols_s2 if c in binary_cols_s2]
    
    num_pipe_s2 = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', RobustScaler())])
    cat_pipe_s2 = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('passthrough', 'passthrough')])

    preprocessor_s2 = ColumnTransformer([
        ('num', num_pipe_s2, cont_cols_s2),
        ('cat', cat_pipe_s2, bin_cols_s2_curr)
    ], verbose_feature_names_out=False)
    preprocessor_s2.set_output(transform='pandas')
    
    X_train_final = preprocessor_s2.fit_transform(X_s2_train)
    X_val_final = preprocessor_s2.transform(X_s2_val)
    X_ext_final = preprocessor_s2.transform(X_s2_ext)
    
    y_val = y_dev[val_idx]
    
    # --- 5. Stacking Training ---
    print("  [Step 3] Training & Optimizing Stacking Model...")
    pos_weight = np.sum(y_train==0) / np.sum(y_train==1)
    
    def optimize(est, params):
        search = RandomizedSearchCV(est, params, n_iter=100, scoring='roc_auc', 
                                    cv=5, n_jobs=-1, random_state=SEED, verbose=0)
        search.fit(X_train_final, y_train)
        return search.best_estimator_

    # 参数空间定义
    xgb_params = {
        'n_estimators': [300, 500, 700],
        'learning_rate': [0.01],
        'max_depth': [3, 4, 5],
        'subsample': [0.7, 0.8, 0.9],
        'colsample_bytree': [0.7, 0.8, 0.9],
        'min_child_weight': [2, 3, 4],
        'scale_pos_weight': [scale_pos_weight]
    }

    lgb_params = {
        'n_estimators': [300, 500, 700],
        'num_leaves': [15, 31],
        'learning_rate': [0.01],
        'max_depth': [4, 5, 6],
        'subsample': [0.7, 0.8, 0.9],
        'min_child_samples': [10, 20],
        'scale_pos_weight': [scale_pos_weight]
    }

    rf_params = {
        'n_estimators': [300, 500, 700],
        'max_depth': [5, 10],
        'min_samples_split': [3, 4, 5],
        'min_samples_leaf': [2, 3, 4],
        'max_features': ['sqrt'],
        'class_weight': ['balanced']
    }

    svm_params = {
        'C': [0.5, 1.0, 2.0],
        'kernel': ['rbf', 'linear'],
        'gamma': ['scale', 'auto']
    }

    best_xgb = optimize_model(xgb.XGBClassifier(eval_metric='auc', random_state=SEED, n_jobs=1), 
                            xgb_params, "XGBoost")
    best_lgbm = optimize_model(lgb.LGBMClassifier(objective='binary', metric='auc', verbosity=-1, random_state=SEED, n_jobs=1), 
                            lgb_params, "LightGBM")
    best_rf = optimize_model(RandomForestClassifier(random_state=SEED, n_jobs=-1), 
                            rf_params, "Random Forest")
    best_svm = optimize_model(SVC(probability=True, random_state=SEED), 
                            svm_params, "SVM")
    best_gnb = GaussianNB(var_smoothing=1e-9)
    
    stacking_clf = StackingClassifier(
        estimators=[('xgb', best_xgb), ('lgbm', best_lgbm), ('rf', best_rf), ('svm', best_svm), ('gnb', best_gnb)],
        final_estimator=LogisticRegression(C=1.0, random_state=SEED), cv=5, n_jobs=-1, passthrough=False
    )
    stacking_clf.fit(X_train_final, y_train)
    
    # --- 6. 评估结果 (Bootstrap 95% CI for BOTH Internal and External) ---
    
    # 寻找最佳阈值
    val_probs_temp = stacking_clf.predict_proba(X_val_final)[:, 1]
    p, r, t = precision_recall_curve(y_val, val_probs_temp)
    f1_curve = 2*(p*r)/(p+r+1e-8)
    best_thresh = t[np.argmax(f1_curve)]
    print(f"  -> Best Threshold found: {best_thresh:.4f}")
    
    # A. 内部验证集 (Internal Val) - 加上 Bootstrap
    print("  [Eval] Bootstrapping Internal Validation...")
    int_str, int_raw, _ = get_bootstrap_stats(stacking_clf, X_val_final, y_val, best_thresh)
    
    # B. 外部测试集 (External Test) - 加上 Bootstrap
    print("  [Eval] Bootstrapping External Test...")
    ext_str, ext_raw, ext_probs = get_bootstrap_stats(stacking_clf, X_ext_final, y_ext, best_thresh)
    
    return {
        'internal_str': int_str,
        'internal_raw': int_raw,
        'external_str': ext_str,
        'external_raw': ext_raw,
        'y_ext_true': y_ext,
        'y_ext_prob': ext_probs
    }

# ==========================================
# 5. 执行实验
# ==========================================
df_dev_0 = df_dev_all[df_dev_all['T'] == 0].copy()
df_dev_1 = df_dev_all[df_dev_all['T'] == 1].copy()
df_test_0 = df_test_all[df_test_all['T'] == 0].copy()
df_test_1 = df_test_all[df_test_all['T'] == 1].copy()

res_A = run_two_stage_pipeline(df_dev_0, df_test_0, "Model A (Control T=0)", use_lac_in_s1=False)
res_B = run_two_stage_pipeline(df_dev_1, df_test_1, "Model B (Treatment T=1)", use_lac_in_s1=True)

# ==========================================
# 6. 结果汇总与统计检验
# ==========================================
def print_stat_comparison(title, res_a, res_b, metric_key_str, metric_key_raw):
    print(f"\n>>> {title} <<<")
    
    # 计算 AUC P-value
    auc_p, auc_z = calculate_p_value(res_a[metric_key_raw]['AUC'], res_b[metric_key_raw]['AUC'])
    
    print(f"{'Metric':<10} | {'Model A (Control)':<30} | {'Model B (Treatment)':<30} | {'P-Value (Z-test)':<20}")
    print("-" * 100)
    
    metrics = ['AUC', 'Accuracy', 'Precision', 'Recall', 'F1']
    
    for m in metrics:
        val_a = res_a[metric_key_str][m]
        val_b = res_b[metric_key_str][m]
        
        # 仅对 AUC 显示 P 值 (当然也可以对其他指标算)
        p_text = ""
        if m == 'AUC':
            sig_mark = "**" if auc_p < 0.05 else ""
            p_text = f"{auc_p:.4f} {sig_mark}"
        
        print(f"{m:<10} | {val_a:<30} | {val_b:<30} | {p_text:<20}")
    print("-" * 100)
    if auc_p < 0.05:
        print("结论: 两个模型在 AUC 上存在统计学显著差异 (P < 0.05)。")
    else:
        print("结论: 两个模型在 AUC 上无统计学显著差异 (P >= 0.05)。")

print("\n" + "="*80)
print("FINAL STATISTICAL REPORT (Mean [95% CI])")
print("="*80)

print_stat_comparison("INTERNAL VALIDATION COMPARISON", res_A, res_B, 'internal_str', 'internal_raw')
print_stat_comparison("EXTERNAL TEST COMPARISON", res_A, res_B, 'external_str', 'external_raw')

# ==========================================
# 7. 可视化 (外部测试集)
# ==========================================
# 使用外部测试集数据
y_true_a, y_prob_a = res_A['y_ext_true'], res_A['y_ext_prob']
y_true_b, y_prob_b = res_B['y_ext_true'], res_B['y_ext_prob']

fig, axes = plt.subplots(1, 3, figsize=(18, 6), dpi=120)

# --- A. ROC Curve ---
fpr_a, tpr_a, _ = roc_curve(y_true_a, y_prob_a)
auc_a = np.mean(res_A['external_raw']['AUC'])
fpr_b, tpr_b, _ = roc_curve(y_true_b, y_prob_b)
auc_b = np.mean(res_B['external_raw']['AUC'])

axes[0].plot(fpr_a, tpr_a, color='#1f77b4', lw=2, label=f'Model A (AUC={auc_a:.3f})')
axes[0].plot(fpr_b, tpr_b, color='#d62728', lw=2, label=f'Model B (AUC={auc_b:.3f})')
axes[0].plot([0, 1], [0, 1], linestyle=':', color='gray')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve (External Test)')
axes[0].legend(loc='lower right')
axes[0].grid(True, linestyle=':', alpha=0.6)

# --- B. Calibration Curve ---
prob_true_a, prob_pred_a = calibration_curve(y_true_a, y_prob_a, n_bins=10, strategy='uniform')
prob_true_b, prob_pred_b = calibration_curve(y_true_b, y_prob_b, n_bins=10, strategy='uniform')
brier_a = brier_score_loss(y_true_a, y_prob_a)
brier_b = brier_score_loss(y_true_b, y_prob_b)

axes[1].plot(prob_pred_a, prob_true_a, marker='o', color='#1f77b4', label=f'Model A (Brier={brier_a:.3f})')
axes[1].plot(prob_pred_b, prob_true_b, marker='s', color='#d62728', label=f'Model B (Brier={brier_b:.3f})')
axes[1].plot([0, 1], [0, 1], linestyle=':', color='gray', label='Perfectly Calibrated')
axes[1].set_xlabel('Mean Predicted Probability')
axes[1].set_ylabel('Fraction of Positives')
axes[1].set_title('Calibration Curve')
axes[1].legend(loc='best')
axes[1].grid(True, linestyle=':', alpha=0.6)

# --- C. Decision Curve Analysis (DCA) ---
thresholds = np.linspace(0.01, 0.99, 100)
nb_a = calculate_net_benefit(y_true_a, y_prob_a, thresholds)
nb_b = calculate_net_benefit(y_true_b, y_prob_b, thresholds)

# Calculate "Treat All"
prevalence = np.mean(y_true_a) # Assuming prevalence is similar
nb_all = prevalence - (1 - prevalence) * (thresholds / (1 - thresholds))
# "Treat None" is 0

axes[2].plot(thresholds, nb_a, color='#1f77b4', lw=2, label='Model A')
axes[2].plot(thresholds, nb_b, color='#d62728', lw=2, label='Model B')
axes[2].plot(thresholds, nb_all, linestyle='--', color='gray', alpha=0.8, label='Treat All')
axes[2].axhline(y=0, color='black', linestyle=':', label='Treat None')

y_min = np.min([np.min(nb_a), np.min(nb_b), -0.1])
y_max = np.max([np.max(nb_a), np.max(nb_b), prevalence]) + 0.05
axes[2].set_ylim(max(y_min, -0.2), y_max)
axes[2].set_xlabel('Threshold Probability')
axes[2].set_ylabel('Net Benefit')
axes[2].set_title('Decision Curve Analysis (DCA)')
axes[2].legend(loc='upper right')
axes[2].grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()